# Deep Learning - LSTM

Unlike TF-IDF which treats each word independently, an LSTM reads text sequentially and retains context from previous words. The bidirectional wrapper processes the sequence in both directions, giving the model a fuller understanding of each word's context.

The text was tokenized and padded to a fixed length of 150 tokens. Words not seen during training are handled with 00V (Out of Vocabulary) token to avoid errors during inference.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

In [2]:
df = pd.read_csv(r"C:\Users\kumri\Downloads\Data Science Notebooks\AirBnB sentiment analysis\sentiment_tags.csv")
print(df.shape)
df.head()

(15116, 10)


,date,reviewer_name,comments,language,exclamation_count,question_count,caps_ratio,word_count,cleaned_comments,sentiment
0,1/18/2020,Santiago,Beca its very lovely! 10/10 recommended!,en,2,0,0.025000,6,beca it very lovely! / recommended!,positive
1,8/10/2020,Serena,Renate is very nice and ready to help. The hou...,en,1,0,0.025381,38,renate be very nice and ready to help. the hou...,positive
2,5/21/2022,Rosemary,"Overall the location was excellent, but the co...",en,0,0,0.005682,60,"overall the location be excellent, but the con...",neutral
3,12/11/2019,Ali,"Highly recommended, good value of money a litt...",en,0,0,0.009804,18,"highly recommended, good value of money a litt...",positive
4,8/27/2016,Zhibo,Suze's house is situated in a very nice and qu...,en,0,0,0.022535,59,suze's house be situate in a very nice and qui...,positive


In [3]:
vocab_size = 10000
max_len = 150

tokenizer = Tokenizer(num_words=vocab_size, oov_token='00V')
tokenizer.fit_on_texts(df['cleaned_comments'])

sequences = tokenizer.texts_to_sequences(df['cleaned_comments'])

padded = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')
print("Forma del dataset:",padded.shape)


Forma del dataset: (15116, 150)


In [4]:
#Encoding sentiment tags
le = LabelEncoder()
y = le.fit_transform(df['sentiment'])

#SMOTE
smote = SMOTE(random_state=42)
padded_balanced, y_balanced = smote.fit_resample(padded, y)
print("Distribution after SMOTE:")
print(pd.Series(y_balanced).value_counts())

#Training
X_train, X_test, y_train, y_test = train_test_split(padded_balanced, 
                                                    y_balanced,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y_balanced
                                                   )

print('Training:',X_train.shape)
print('Test:', X_test.shape)

Distribution after SMOTE:
2    6801
1    6801
0    6801
Name: count, dtype: int64
Training: (16322, 150)
Test: (4081, 150)


In [5]:
#Building model
model = Sequential([
    Embedding(vocab_size, 128),
    Bidirectional(LSTM(64, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

Epoch 1/5
460/460 ━━━━━━━━━━━━━━━━━━━━ 21s 40ms/step - accuracy: 0.7502 - loss: 0.6131 - val_accuracy: 0.8524 - val_loss: 0.3878
Epoch 2/5
460/460 ━━━━━━━━━━━━━━━━━━━━ 17s 36ms/step - accuracy: 0.8619 - loss: 0.3636 - val_accuracy: 0.8524 - val_loss: 0.3738
Epoch 3/5
460/460 ━━━━━━━━━━━━━━━━━━━━ 16s 36ms/step - accuracy: 0.9005 - loss: 0.2730 - val_accuracy: 0.8592 - val_loss: 0.3652
Epoch 4/5
460/460 ━━━━━━━━━━━━━━━━━━━━ 17s 36ms/step - accuracy: 0.9267 - loss: 0.2118 - val_accuracy: 0.8592 - val_loss: 0.3908
Epoch 5/5
460/460 ━━━━━━━━━━━━━━━━━━━━ 17s 37ms/step - accuracy: 0.9425 - loss: 0.1713 - val_accuracy: 0.8481 - val_loss: 0.4535


## Results

The LSTM achieved 86% overall accuracy with balanced performance across all three sentiment classes after applying SMOTE. However, Linear SVC remains the stronger model overall particularly for negative review detection, where it outperforms the LSTM in both precision and recall.

In [7]:
y_pred = np.argmax(model.predict(X_test), axis=1)

print(classification_report(y_test, y_pred, target_names=le.classes_))

128/128 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step
              precision    recall  f1-score   support

    negative       0.87      0.84      0.85      1360
     neutral       0.82      0.84      0.83      1360
    positive       0.89      0.91      0.90      1361

    accuracy                           0.86      4081
   macro avg       0.86      0.86      0.86      4081
weighted avg       0.86      0.86      0.86      4081



In [8]:
model.save('lstm_model.keras')
import joblib
joblib.dump(tokenizer,'lstm_tokenizer.pkl')
print('Model save')

Model save
